In [0]:
%load_ext autoreload
%autoreload 2
# Enables autoreload; learn more at https://docs.databricks.com/en/files/workspace-modules.html#autoreload-for-python-modules
# To disable autoreload; run %autoreload 0

In [0]:
from calc_functions import *

In [0]:
%sql
CREATE TABLE dev.mohit_gangwani.ad_viewing_presence_opportunities_dma_station_viewing_type AS
WITH minute_bins AS (
    SELECT TIMESTAMP_SECONDS(DIV(UNIX_SECONDS(vc.session_start), 600) * 600) AS time_bin
    , vc.fk_dma_id
    , CASE WHEN st.station_id IS NOT NULL THEN st.local_or_national
           WHEN inp.reported_input_source = 'ANTENNA' THEN 'Local'
           ELSE 'Unknown'
      END AS station_type
    , CASE WHEN st.station_id IS NOT NULL OR inp.reported_input_source = 'ANTENNA' THEN 'Linear'
           WHEN NVL(vc.prev_vizio_epg_station, vc.next_vizio_epg_station) IS NOT NULL OR inp.reported_input_source = 'APPS' THEN 'APPS'
           ELSE 'Unknown'
      END AS app_or_linear
    , vc.fk_tvid
    FROM prod.detection.viewing_commercials_firehose_dedup vc
    LEFT JOIN prod.detection.epg_station st
      ON st.station_id = NVL(vc.prev_station_id, vc.next_station_id)
     AND st.vendor_name = 'TIVO'
    JOIN prod.detection.commercial_id_external_firehose cief
      ON cief.external_id = vc.external_id
    JOIN prod.detection.clients cl
      ON cl.client_id = cief.fk_client_id
    LEFT JOIN prod.detection.input_source inp
      ON inp.input_source_id = vc.fk_input_source_id
    WHERE vc.session_start >= CURRENT_DATE - 7
      AND vc.session_start < CURRENT_DATE
      AND vc.fk_zoo_id = 17
      AND cl.client_name = 'kinetiq'
      AND vc.fk_dma_id IS NOT NULL
    GROUP BY ALL
)
SELECT time_bin
, fk_dma_id
, station_type
, app_or_linear
, COUNT(DISTINCT fk_tvid) AS active_tvs
FROM minute_bins
GROUP BY 1,2,3,4;

In [0]:
%sql
CREATE TABLE dev.mohit_gangwani.ad_viewing_presence_pi_dma_startum AS
WITH opp_7d AS (
  SELECT fk_dma_id
  , station_type
  , app_or_linear
  , SUM(active_tvs) AS opp_7d
  FROM dev.mohit_gangwani.ad_viewing_presence_opportunities_dma_station_viewing_type
  GROUP BY 1,2,3
)
, opp_totals AS (
  SELECT station_type
  , app_or_linear
  , SUM(opp_7d) AS opp_7d_total
  FROM opp_7d
  GROUP BY 1,2
)
SELECT o.fk_dma_id
, o.station_type
, o.app_or_linear
, o.opp_7d
, t.opp_7d_total
, CASE WHEN GREATEST(t.opp_7d_total, 0) = 0 THEN NULL ELSE o.opp_7d/t.opp_7d_total END AS pi_dma_stratum
FROM opp_7d o
JOIN opp_totals t
ON o.station_type = t.station_type
AND o.app_or_linear = t.app_or_linear;

In [0]:
%sql
SELECT * FROM dev.mohit_gangwani.ad_viewing_presence_pi_dma_startum

In [0]:
%sql
DROP TABLE IF EXISTS dev.mohit_gangwani.ad_viewing_renormalized_within;
CREATE TABLE dev.mohit_gangwani.ad_viewing_baseline_pi_weekly AS
WITH pi_raw AS (
  SELECT fk_dma_id
  , station_type
  , app_or_linear
  , pi_dma_stratum
  FROM dev.mohit_gangwani.ad_viewing_presence_pi_dma_startum
)
, pi_eps AS (
  SELECT fk_dma_id
  , station_type
  , app_or_linear
  , GREATEST(pi_dma_stratum, 1e-8) AS pi_eps
  FROM pi_raw
),
renorm AS (
  SELECT station_type
  , app_or_linear
  , SUM(pi_eps) AS sum_pi_eps
  FROM pi_eps
  GROUP BY 1,2
)
SELECT p.fk_dma_id
, p.station_type
, p.app_or_linear
, p.pi_eps / r.sum_pi_eps AS pi_dma_stratum_stable
FROM pi_eps p
JOIN renorm r
  ON p.station_type = r.station_type
 AND p.app_or_linear = r.app_or_linear;

In [0]:
%sql
SELECT * FROM dev.mohit_gangwani.ad_viewing_baseline_pi_weekly;

In [0]:
%sql
DROP TABLE IF EXISTS dev.mohit_gangwani.ad_viewing_type_dma_weekly;
CREATE TABLE dev.mohit_gangwani.ad_viewing_type_dma_weekly AS
SELECT vc.external_id AS ad_id
, CASE WHEN st.station_id IS NOT NULL THEN st.local_or_national
           WHEN inp.reported_input_source = 'ANTENNA' THEN 'Local'
           ELSE 'Unknown'
      END AS station_type
, CASE WHEN st.station_id IS NOT NULL OR inp.reported_input_source = 'ANTENNA' THEN 'Linear'
        WHEN NVL(vc.prev_vizio_epg_station, vc.next_vizio_epg_station) IS NOT NULL OR inp.reported_input_source = 'APPS' THEN 'APPS'
        ELSE 'Unknown'
  END AS app_or_linear
, vc.fk_dma_id
, COUNT(DISTINCT vc.fk_tvid) AS tv_count
, COUNT(DISTINCT vc.fk_tvid||'_'||vc.session_start) AS impression_count
FROM prod.detection.viewing_commercials_firehose_dedup vc
LEFT JOIN prod.detection.epg_station st
  ON st.station_id = NVL(vc.prev_station_id, vc.next_station_id)
 AND st.vendor_name = 'TIVO'
JOIN prod.detection.commercial_id_external_firehose cief
  ON cief.external_id = vc.external_id
JOIN prod.detection.clients cl
  ON cl.client_id = cief.fk_client_id
LEFT JOIN prod.detection.input_source inp
  ON inp.input_source_id = vc.fk_input_source_id
WHERE vc.session_start >= CURRENT_DATE - 7
  AND vc.session_start < CURRENT_DATE
  AND vc.fk_zoo_id = 17
  AND cl.client_name = 'kinetiq'
  AND vc.fk_dma_id IS NOT NULL
GROUP BY 1, 2, 3, 4
HAVING tv_count >= 10
   AND impression_count >= 20
;

In [0]:
%sql
DROP TABLE IF EXISTS dev.mohit_gangwani.ad_viewing_corrected_viewing_bias;
CREATE TABLE dev.mohit_gangwani.ad_viewing_corrected_viewing_bias AS
WITH ad_dma AS (
  SELECT ad_id
  , fk_dma_id
  , station_type
  , app_or_linear
  , SUM(impression_count) AS total_impression_count
  FROM dev.mohit_gangwani.ad_viewing_type_dma_weekly
  GROUP BY 1,2,3,4
)
, ad_tot AS (
  SELECT ad_id
  , SUM(total_impression_count) AS ad_imps_total
  FROM ad_dma
  GROUP BY 1
)
, ad_p AS (
  SELECT a.ad_id
  , a.fk_dma_id
  , a.station_type
  , a.app_or_linear
  , a.total_impression_count
  , t.ad_imps_total
  , a.total_impression_count/t.ad_imps_total AS p_raw
  FROM ad_dma a
  JOIN ad_tot t
    ON a.ad_id = t.ad_id
)
, joined AS (
  SELECT p.*
  , p.ad_imps_total
  , p.total_impression_count
  , b.pi_dma_stratum_stable AS pi
  FROM ad_p p
  LEFT JOIN dev.mohit_gangwani.ad_viewing_baseline_pi_weekly b
    ON p.fk_dma_id = b.fk_dma_id 
   AND p.station_type = b.station_type
   AND p.app_or_linear = b.app_or_linear
)
, bias_corrected AS (
  SELECT ad_id
  , fk_dma_id
  , station_type
  , app_or_linear
  , ad_imps_total
  , total_impression_count
  , p_raw
  , pi
  , p_raw/NULLIF(pi, 0) AS w
  FROM joined
)
, renorm AS (
  SELECT ad_id
  , SUM(w) AS w_sum
  FROM bias_corrected
  GROUP BY 1
)
SELECT b.ad_id
, b.fk_dma_id
, b.station_type
, b.app_or_linear
, b.ad_imps_total
, b.total_impression_count
, b.w/r.w_sum AS p_bias_corrected
FROM bias_corrected b
JOIN renorm r
  ON b.ad_id = r.ad_id;

In [0]:
%sql
SELECT * FROM dev.mohit_gangwani.ad_viewing_corrected_viewing_bias
ORDER BY total_impression_count DESC, p_bias_corrected DESC
LIMIT 1000

In [0]:
df_from_sql = spark.sql("""
        WITH ovrl AS (   
        SELECT ad_id, SUM(total_impression_count) AS ttl_count
        FROM dev.mohit_gangwani.ad_viewing_corrected_viewing_bias
        -- WHERE viewing_type != 'Unknown'
        GROUP BY 1
        )
        , ad_filter AS (
        SELECT ad_id, ttl_count, DENSE_RANK() OVER (ORDER BY ttl_count DESC) AS rk
        FROM ovrl
        )
        SELECT a.*
        FROM dev.mohit_gangwani.ad_viewing_corrected_viewing_bias a
        JOIN ad_filter f
        ON f.ad_id = a.ad_id
        WHERE f.rk <= 10000""")
vdf = df_from_sql.toPandas()
vdf.head(20)

In [0]:
v_agg_df = vdf.groupby(['ad_id']).agg(
    normalized_gini=('p_bias_corrected', lambda x: normalized_gini(x)),
    normalized_entropy=('p_bias_corrected', lambda x: normalized_entropy(x)),
    locality_index=('p_bias_corrected', lambda x: 1 - normalized_entropy(x)),
).reset_index()

v_agg_df.head(10)

In [0]:
scatterplot(df=v_agg_df, x='normalized_gini', y='locality_index', title='Gini vs Locality Index', xlabel='Gini', ylabel='Locality Index')

In [0]:
features = vdf.groupby(['ad_id']).agg(
    normalized_gini=('p_bias_corrected', lambda x: normalized_gini(x)),
    normalized_entropy=('p_bias_corrected', lambda x: normalized_entropy(x)),
    locality_index=('p_bias_corrected', lambda x: 1 - normalized_entropy(x)),
).reset_index()

# features = vdf.groupby('ad_id')['p_bias_corrected'].apply(lambda x: pd.Series({
#     'normalized_entropy': normalized_entropy(x),
#     'locality index': 1 - normalized_entropy(x),
#     'normalized_gini': normalized_gini(x)
# })).reset_index()

In [0]:
features.head()

In [0]:
df_plot = features.merge(vdf, on = 'ad_id')

In [0]:
df_plot.head()

In [0]:
sns.scatterplot(data=df_plot, x = 'normalized_gini', y='locality_index', hue='station_type', alpha = 0.25, s=10)
plt.title('Footprint Shape: Concetration v Uniformity')
plt.xlabel('Normalized Gini (Concentration)')
plt.ylabel('Locality Index (Uniformity: 1 - Entropy)')
plt.show()

In [0]:
sns.scatterplot(data=df_plot, x = 'normalized_gini', y='locality_index', hue='app_or_linear', alpha = 0.5, s=10)
plt.title('Footprint Shape: Concetration v Uniformity')
plt.xlabel('Normalized Gini (Concentration)')
plt.ylabel('Locality Index (Uniformity: 1 - Entropy)')
plt.show()

In [0]:
ax = sns.kdeplot(data=df_plot, x = 'normalized_gini', y='locality_index', hue='station_type', alpha = 0.25)
sns.move_legend(ax, "upper left")
plt.title('Footprint Shape: Concetration v Uniformity')
plt.xlabel('Normalized Gini (Concentration)')
plt.ylabel('Locality Index (Uniformity: 1 - Entropy)')
plt.show()

In [0]:
ax = sns.kdeplot(data=df_plot, x = 'normalized_gini', y='locality_index', hue='app_or_linear', alpha = 0.25)
sns.move_legend(ax, "upper left")
plt.title('Footprint Shape: Concetration v Uniformity')
plt.xlabel('Normalized Gini (Concentration)')
plt.ylabel('Locality Index (Uniformity: 1 - Entropy)')
plt.show()

In [0]:
%sql
WITH opp_per_dma AS (
  SELECT fk_dma_id, SUM(active_tvs) AS opp
  FROM dev.mohit_gangwani.ad_viewing_presence_opportunities_dma_station_viewing_type
  GROUP BY 1
)
, tot AS (
  SELECT SUM(opp) AS opp_total FROM opp_per_dma
)
SELECT COUNT(*) AS num_dmas
, MIN(opp) AS min_opp
, MAX(opp) AS max_opp
, MAX(opp)/MIN(NULLIF(opp,0)) AS max_min_ratio
-- , APPROX_TOP_COUNT(fk_dma_id, 5) AS top5_dmas_by_opp
, APPROX_PERCENTILE(opp, 0.5) AS median_opp
FROM opp_per_dma;

In [0]:

%sql
-- Top-k share (top 5 & top 10 DMAs)
WITH opp_per_dma AS (
  SELECT fk_dma_id, SUM(active_tvs) AS opp
  FROM dev.mohit_gangwani.ad_viewing_presence_opportunities_dma_station_viewing_type
  GROUP BY 1
)
, ranked AS (
  SELECT fk_dma_id, opp
  , DENSE_RANK() OVER (ORDER BY opp DESC) AS r
  FROM opp_per_dma
)
, tot AS (
  SELECT SUM(opp) AS opp_total FROM opp_per_dma
)
SELECT SUM(CASE WHEN r <= 5 THEN opp ELSE 0 END)/(SELECT opp_total FROM tot) AS top5_share
, SUM(CASE WHEN r <= 10 THEN opp ELSE 0 END)/(SELECT opp_total FROM tot) AS top10_share
FROM ranked

In [0]:
%sql
WITH pi AS ( SELECT COALESCE(station_type, 'UNKNOWN') AS station_type
, COALESCE(app_or_linear, 'UNKNOWN') AS app_or_linear
, fk_dma_id
, pi_dma_stratum_stable AS pi
FROM dev.mohit_gangwani.ad_viewing_baseline_pi_weekly
WHERE pi_dma_stratum_stable IS NOT NULL
AND pi_dma_stratum_stable > 0
)
-- Rank DMAs by share (ascending) and get group sizes
,
ranked AS (
  SELECT station_type
  , app_or_linear
  , fk_dma_id
  , pi
  , ROW_NUMBER() OVER (PARTITION BY station_type, app_or_linear ORDER BY pi ASC) AS i
  , COUNT(*) OVER (PARTITION BY station_type, app_or_linear) AS n
  , SUM(pi) OVER (PARTITION BY station_type, app_or_linear ) AS sum_pi
  FROM pi
)
-- Gini (raw and normalized)
, gini AS (
  SELECT station_type
  , app_or_linear
  , CASE WHEN MIN(n) = 1 THEN 0.0 ELSE (2.0 * SUM(i * pi) / (MAX(n) * MAX(sum_pi))) - ((MAX(n) + 1.0) / MAX(n)) END AS gini_raw
  , CASE WHEN MIN(n) = 1 THEN 0.0 ELSE (MAX(n)/(MAX(n) - 1.0)) * ((2.0 * SUM(i * pi) / (MAX(n) * MAX(sum_pi))) - ((MAX(n) + 1.0) / MAX(n))) END AS gini_normalized
  FROM ranked
  GROUP BY 1, 2
)
-- HHI and Effective # of DMAs
, hhi AS (
  SELECT station_type
  , app_or_linear
  , SUM(pi * pi) AS hhi, 1.0 / SUM(pi * pi) AS effective_dmas
  FROM pi
  GROUP BY 1, 2
)
SELECT g.station_type
, g.app_or_linear
, g.gini_raw
, g.gini_normalized
, h.hhi
, h.effective_dmas
FROM gini g
JOIN hhi h
  ON g.station_type = h.station_type
 AND g.app_or_linear = h.app_or_linear
ORDER BY g.station_type, g.app_or_linear;

In [0]:
%sql
WITH pi AS (
  SELECT fk_dma_id
  , SUM(pi_dma_stratum_stable) AS pi
  FROM dev.mohit_gangwani.ad_viewing_baseline_pi_weekly
  WHERE pi_dma_stratum_stable IS NOT NULL
    AND pi_dma_stratum_stable > 0
  GROUP BY 1
),
ranked AS (
  SELECT fk_dma_id
  , pi
  , ROW_NUMBER() OVER (ORDER BY pi ASC) AS i
  , COUNT(*) OVER () AS n
  , SUM(pi) OVER () AS sum_pi
  FROM pi
)
SELECT CASE WHEN MIN(n)=1 THEN 0.0 ELSE (2.0 * SUM(i*pi) / (MAX(n)*MAX(sum_pi))) - ((MAX(n)+1.0)/MAX(n)) END AS gini_raw
, CASE WHEN MIN(n)=1 THEN 0.0 ELSE (MAX(n)/(MAX(n)-1.0)) * ((2.0 * SUM(i*pi) / (MAX(n)*MAX(sum_pi))) - ((MAX(n)+1.0)/MAX(n))) END AS gini_normalized
, SUM(pi*pi) AS hhi
, 1.0 / SUM(pi*pi) AS effective_dmas
FROM ranked;